# Imports

In [189]:
from datasets import load_dataset
from datasets import get_dataset_split_names
import tiktoken
from pathlib import Path
import json
from typing import Dict, List, Any
import numpy as np


# Download TinyStories
- load_dataset: https://huggingface.co/docs/datasets/en/load_hub
- dataset: https://huggingface.co/docs/datasets/en/access
- TinyStories: https://huggingface.co/datasets/roneneldan/TinyStories

In [13]:
get_dataset_split_names("roneneldan/TinyStories")

['train', 'validation']

In [2]:
dataset = load_dataset("roneneldan/TinyStories")

README.md:   0%|          | 0.00/1.06k [00:00<?, ?B/s]

data/train-00000-of-00004-2d5a1467fff108(…): reconstructing file:   0%|          |  0.00B /  249MB            

data/train-00000-of-00004-2d5a1467fff108(…): downloading bytes:           |  0.00B            

data/train-00001-of-00004-5852b56a2bd28f(…): reconstructing file:   0%|          |  0.00B /  248MB            

data/train-00001-of-00004-5852b56a2bd28f(…): downloading bytes:           |  0.00B            

...

data/train-00003-of-00004-d243063613e5a0(…): downloading bytes:           |  0.00B            

data/validation-00000-of-00001-869c898b5(…): reconstructing file:   0%|          |  0.00B / 9.99MB            

data/validation-00000-of-00001-869c898b5(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2119719 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/21990 [00:00<?, ? examples/s]

In [6]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 2119719
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 21990
    })
})

In [14]:
dataset['train']

Dataset({
    features: ['text'],
    num_rows: 2119719
})

In [15]:
dataset['train'][0]

{'text': 'One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.\n\nLily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."\n\nTogether, they shared the needle and sewed the button on Lily\'s shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.'}

# Create sample data

In [80]:
total_samples = 5

samples = []
for i in range(total_samples):
    samples.append(dataset['train'][i]['text'])

In [81]:
samples

['One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.\n\nLily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."\n\nTogether, they shared the needle and sewed the button on Lily\'s shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.',
 'Once upon a time, there was a little car named Beep. Beep loved to go fast and play in the sun. Beep was a healthy car because he always had good fuel. Good fuel made Beep happy and strong.\n\nOne day, Beep was driving in the park when he saw a big tree. The tree had many leaves that we

In [83]:
break_cond = 5
for i, story in enumerate(dataset['train']['text']):
    print(i, '\n', story)
    print('-'*50)
    if i>=break_cond:
        break

0 
 One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.

Lily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."

Together, they shared the needle and sewed the button on Lily's shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.
--------------------------------------------------
1 
 Once upon a time, there was a little car named Beep. Beep loved to go fast and play in the sun. Beep was a healthy car because he always had good fuel. Good fuel made Beep happy and strong.

One day, Beep was driving in the park when he sa

# GPT tokenizer

In [36]:
enc = tiktoken.get_encoding("gpt2")
print('enc.eot_token:', enc.eot_token)
print('enc._special_tokens:', enc._special_tokens)
print('enc.decode([50256]):', enc.decode([50256]))



enc.eot_token: 50256
enc._special_tokens: {'<|endoftext|>': 50256}
enc.decode([50256]): <|endoftext|>


In [78]:
enc.encode('<EOS>')

[27, 36, 2640, 29]

# Save file in .txt format
- Not a good idea. The escape sequences like "\n\n" are part of the data and present in each story. 
- If we dump this as a massive text file the distinction b/w the stories are lost. 
    - We can counter this by adding a "<|endoftext|>" after each story - and that works well.
    - The problem with this is that it becomes married to the gpt2 tokenizer - if we want to use a sperate tokenizer, "<|endoftext|>" would not make sense

In [87]:
sample_file_path = Path.cwd()/"sample.txt"
sample_file_path

PosixPath('/Users/irabandutta/Developer/2026-08-llm-from-scratch/notebooks/sample.txt')

In [88]:
if sample_file_path.exists():
    print(f'{sample_file_path} already present!')
else:
    print(f'Writing to {sample_file_path}')
    with open(sample_file_path, 'w', encoding='utf-8') as f:
        for story in samples:
            f.write(story)
            # f.write('\n')
    print(f'Writing complete')

Writing to /Users/irabandutta/Developer/2026-08-llm-from-scratch/notebooks/sample.txt
Writing complete


In [89]:
if sample_file_path.exists():
    print(f'{sample_file_path} file already present!')
else:
    print(f'Writing file to {sample_file_path}')
    # with open(sample_file_path, 'w', encoding='utf-8') as f:
    with open(sample_file_path, 'w') as f:
        max_limit=10
        for i, story in enumerate(dataset['train']['text']):
            f.write(story)
            # f.write('\n')
            if i>=max_limit:
                break
    print(f'Writing complete')

/Users/irabandutta/Developer/2026-08-llm-from-scratch/notebooks/sample.txt file already present!


# Save file in .jsonl format


In [90]:
sample_file_path = Path.cwd()/"sample.jsonl"
sample_file_path

PosixPath('/Users/irabandutta/Developer/2026-08-llm-from-scratch/notebooks/sample.jsonl')

In [99]:
if sample_file_path.exists():
    print(f'{sample_file_path} file already present!')
else:
    print(f'Writing file to {sample_file_path}')
    with open(sample_file_path, 'w', encoding='utf-8') as f:
    # with open(sample_file_path, 'w') as f:
        max_limit=10
        for i, story in enumerate(dataset['train']['text']):
            record = {
                'id':i,
                "text":story
            }
            # JSON:
            json.dump(record, f, ensure_ascii=False)
            # JSON: Dump string
            # f.write(json.dumps(record, ensure_ascii=False))
            f.write('\n')
            if i>=max_limit:
                break
    print(f'Writing complete')

Writing file to /Users/irabandutta/Developer/2026-08-llm-from-scratch/notebooks/sample.jsonl
Writing complete


# Reading from .jsonl file

In [117]:
# Option1: Not memory efficient
with open(sample_file_path, 'r', encoding='utf-8') as f:
    # data = f.read()
    # data = f.readline()
    data_lst = f.readlines() # Read all lines as string and returns a list
    data = [json.loads(i)['text'] for i in data_lst]

data

['One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.\n\nLily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."\n\nTogether, they shared the needle and sewed the button on Lily\'s shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.',
 'Once upon a time, there was a little car named Beep. Beep loved to go fast and play in the sun. Beep was a healthy car because he always had good fuel. Good fuel made Beep happy and strong.\n\nOne day, Beep was driving in the park when he saw a big tree. The tree had many leaves that we

### Using Generators (`yield`)

- Consider the following function:

```python
def squares():
    return [1, 4, 9]
```

- When we call `squares()`:
    - Python first computes the **entire list** `[1, 4, 9]` in memory.
    - The entire list is returned to the caller.
    - The function finishes execution.
    - Its execution state is destroyed.

- Now consider a generator:

```python
def squares():
    yield 1
    yield 4
    yield 9
```

- Calling `squares()`:
    - Does **not** execute the function immediately.
    - Returns a **generator object**, which can produce one value at a time.

- Execution proceeds as follows:
    - `g = squares()`
        - Returns a generator object.
        - The function body has **not** executed yet.
    - `next(g) --> 1`
        - The function starts executing.
        - Returns `1`.
        - Execution pauses at the first `yield`.
    - `next(g) --> 4`
        - Execution resumes from the previous `yield`.
        - Returns `4`.
        - Execution pauses at the second `yield`.
    - `next(g) --> 9`
        - Execution resumes again.
        - Returns `9`.
        - Execution pauses at the third `yield`.
    - `next(g) --> StopIteration`
        - There are no more `yield` statements.
        - The generator is exhausted.

- Unlike `return`:
    - `return` exits the function permanently.
    - `yield` returns a value **and pauses the function**, preserving its local variables and current execution state.
    - Each call to `next()` resumes execution exactly from the previous `yield`.

- A `for` loop simply calls `next()` on the generator internally until a `StopIteration` exception is raised.

- This is why generators are memory efficient:
    - They produce one value only when it is requested.
    - They do **not** create and store the entire output in memory upfront.

- The same idea is used when reading a JSONL file:
    - Only one line is loaded into memory.
    - The line is parsed and processed.
    - The line is discarded.
    - The next line is then read.
    - As a result, memory usage remains nearly constant whether the dataset contains **10 stories or 10 million stories**.

In [123]:
# Option2: Memory efficient

def iter_jsonl(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            yield json.loads(line)

data = []
for sample in iter_jsonl(sample_file_path):
    data.append(sample['text'])

    # Can call inference for 1 sample or a batch of samples while keep the generator in paused staet until next(g) is called again
    # Can call tokenizer as fell on 1 sample at a time

data

['One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.\n\nLily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."\n\nTogether, they shared the needle and sewed the button on Lily\'s shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.',
 'Once upon a time, there was a little car named Beep. Beep loved to go fast and play in the sun. Beep was a healthy car because he always had good fuel. Good fuel made Beep happy and strong.\n\nOne day, Beep was driving in the park when he saw a big tree. The tree had many leaves that we

In [126]:
sample_file_path.stat().st_size

8406

# Read .jsonl file, tokenize text and save raw binary file

In [136]:
enc = tiktoken.get_encoding("gpt2")

In [143]:
print('# Words:', len(data[0].split()))
print('-'*100)
print(data[0])

# Words: 134
----------------------------------------------------------------------------------------------------
One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.

Lily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."

Together, they shared the needle and sewed the button on Lily's shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.


In [146]:
print('# Tokens:', len(enc.encode(data[0])))
print('-'*100)
print(enc.encode(data[0]))

# Tokens: 162
----------------------------------------------------------------------------------------------------
[3198, 1110, 11, 257, 1310, 2576, 3706, 20037, 1043, 257, 17598, 287, 607, 2119, 13, 1375, 2993, 340, 373, 2408, 284, 711, 351, 340, 780, 340, 373, 7786, 13, 20037, 2227, 284, 2648, 262, 17598, 351, 607, 1995, 11, 523, 673, 714, 34249, 257, 4936, 319, 607, 10147, 13, 198, 198, 43, 813, 1816, 284, 607, 1995, 290, 531, 11, 366, 29252, 11, 314, 1043, 428, 17598, 13, 1680, 345, 2648, 340, 351, 502, 290, 34249, 616, 10147, 1701, 2332, 1995, 13541, 290, 531, 11, 366, 5297, 11, 20037, 11, 356, 460, 2648, 262, 17598, 290, 4259, 534, 10147, 526, 198, 198, 41631, 11, 484, 4888, 262, 17598, 290, 384, 19103, 262, 4936, 319, 20037, 338, 10147, 13, 632, 373, 407, 2408, 329, 606, 780, 484, 547, 7373, 290, 5742, 1123, 584, 13, 2293, 484, 5201, 11, 20037, 26280, 607, 1995, 329, 7373, 262, 17598, 290, 18682, 607, 10147, 13, 1119, 1111, 2936, 3772, 780, 484, 550, 4888, 290, 3111, 1978, 13]


In [149]:
# Check Encode - Decode
enc.decode(enc.encode('One day'))

'One day'

In [150]:
# Check Vocab size
print(enc.n_vocab)

50257


In [159]:
# Since there are ~50k. tokens, to store the tokens, we can use a numpy array
# The datatype of this nupy array can be an unsigned intered of 16 bits
# 2**16 = 65536 which is enough to store ~50k distinct integers representing different tokens

In [217]:
def tokenize_tinystories(story: str, tokenizer_enc: Any) -> list[int]:
    if (
        not hasattr(tokenizer_enc, "eot_token")
        or tokenizer_enc.eot_token != 50256
    ):
        raise ValueError(
            "Expected a tiktoken GPT-2 Encoding object "
            "(eot_token should be 50256)."
        )

    story_tokens = tokenizer_enc.encode(story)
    story_tokens.append(tokenizer_enc.eot_token)
    return story_tokens


def iter_jsonl(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            yield json.loads(line)



# Test 1 story
g = iter_jsonl(sample_file_path)
story_text = next(g)['text']
story_tokens = tokenize_tinystories(story_text, enc)
print(story_tokens[-1])
print('-'*50)

ts_tokens_test = []
for story in iter_jsonl(sample_file_path):
    ts_tokens_test.extend(tokenize_tinystories(story['text'], enc))


print('Total Tokens:', len(ts_tokens_test))


50256
--------------------------------------------------
Total Tokens: 1994


In [221]:
sample_bin_file_path = sample_file_path.parent/'sample.bin'
print(sample_bin_file_path)

batch_size=2
print(f'Tokenizing {sample_file_path.name} and saving in .bin format')
with open(sample_bin_file_path, 'wb') as f:

    ts_tokens_batch = []
    for i, story in enumerate(iter_jsonl(sample_file_path)):
        # Read 1 story at a time from json and append the story tokens in the list
        ts_tokens_batch.extend(tokenize_tinystories(story['text'], enc))

        # Once batch size is reached, we convert te list into a np array and dump imn th binary file
        if i%batch_size==0:
            ts_tokens_batch_np = np.array(ts_tokens_batch, dtype=np.uint16)
            ts_tokens_batch_np.tofile(f)

            ts_tokens_batch = []

    # Write the final partial batch        
    if ts_tokens_batch:
        ts_tokens_batch_np = np.array(ts_tokens_batch, dtype=np.uint16)
        ts_tokens_batch_np.tofile(f)
print(f'Saving complete')

/Users/irabandutta/Developer/2026-08-llm-from-scratch/notebooks/sample.bin
Tokenizing sample.jsonl and saving in .bin format
Saving complete


# Function: download_tinystories

In [ ]:
def download_tinystories(savedir_path:str="data/tinystories", max_limit=None)->Dict:

    print('='*50)
    print('Downloading TinyStories from HF...')
    print('='*50)

    # Define path for tinystories data
    ts_raw_dir_path = Path(savedir_path)/"raw"

    # Create directories
    ts_raw_dir_path.mkdir(parents=True, exists_ok=True)

    # File name for text file
    ts_raw_train_file_path = ts_raw_dir_path/"train.jsonl"
    ts_raw_val_file_path = ts_raw_dir_path/"val.jsonl"

    # Check if file already exists
    if ts_raw_train_file_path.exists():
        print("JSON files exist - download skipped")
    else:
        # Download file
        print('Starting download')
        dataset = load_dataset("roneneldan/TinyStories")

        # Save Training + Validation  file
        for split in ['train', 'validation']:
            
            print(f'Saving .jsonl file for split: {split}')
            if split == "train":
                filepath = ts_raw_train_file_path
            else:
                filepath = ts_raw_val_file_path
            with open(filepath, 'w', encoding='utf-8') as f:
                for i, story in enumerate(dataset[split]):
                    record = {
                        'id':i,
                        "text":story['text']
                    }
                    json.dump(record, f, ensure_ascii=False)
                    f.write('\n')
                    # Track progress
                    if (i+1)%10000==0:
                        print(f'#Samples saved: {i}')
                    if max_limit is not None and i>=max_limit:
                        break
            print(f'Saving complete')


    # File sizes
    train_file_size_mb = ts_raw_train_file_path.stat().st_size/(1024**2)
    val_file_size_mb = ts_raw_val_file_path.stat().st_size/(1024**2)

    print('Downloading TinyStories data complete')
    print(f' Train file path: {ts_raw_train_file_path} | Size: {train_file_size_mb:2f} MB')
    print(f' Val file path: {ts_raw_val_file_path} | Size: {val_file_size_mb:2f} MB')

    return {
        "train": ts_raw_train_file_path,
        "val": ts_raw_val_file_path
    }
            


# Function: tokenize tinystoreis

In [ ]:
def tokenize_tinystories(story: str, tokenizer: Any) -> list[int]:
    if (
        not hasattr(tokenizer, "eot_token")
        or tokenizer.eot_token != 50256
    ):
        raise ValueError(
            "Expected a tiktoken GPT-2 Encoding object "
            "(eot_token should be 50256)."
        )

    story_tokens = tokenizer.encode(story)
    story_tokens.append(tokenizer.eot_token)
    return story_tokens


def iter_jsonl(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            yield json.loads(line)


def tokenize_save_binary(loadfile_path:Path, savefile_name:str, chunk_size:int=100, tokenizer:str='gpt2'):

    print('='*50)
    print(f'Tokenizing {loadfile_path.name} and saving in .bin format')
    print('='*50)

    # Create savefile_dir
    savefile_dir = loadfile_path.parent.parent/f'processed/'
    savefile_dir.mkdir(parents=True, exists_ok=True)

    # Create savefile_path
    savefile_path = savefile_dir/f'{savefile_name}'
    
    if savefile_path.exists():
        print("Binary files exist - tokenizing and saving skipped")
    else:

        # Define Tokenizer
        import tiktoken
        enc=tiktoken.get_encoding(tokenizer)

        # Tokenize and save binary file in chunks
        
        print(f'Tokenizing and saving .bin file chunk by chunk')
        with open(savefile_path, 'wb') as f:

            ts_tokens_chunk = []
            for i, story in enumerate(iter_jsonl(loadfile_path)):
                # Read 1 story at a time from json and append the story tokens in the list
                ts_tokens_chunk.extend(tokenize_tinystories(story['text'], enc))

                # Once batch size is reached, we convert te list into a np array and dump imn th binary file
                if (i+1)%chunk_size==0:
                    ts_tokens_chunk_np = np.array(ts_tokens_chunk, dtype=np.uint16)
                    ts_tokens_chunk_np.tofile(f)

                    ts_tokens_chunk = []

            # Write the final partial batch        
            if ts_tokens_chunk:
                ts_tokens_chunk_np = np.array(ts_tokens_chunk, dtype=np.uint16)
                ts_tokens_chunk_np.tofile(f)
        print(f'Saving complete')

    # File size:
    savefile_size_mb = savefile_path.stat().st_size/(1024**2)
    print('Tokenizing TinyStories and saving binary data complete')
    print(f' File path: {savefile_size_mb} | Size: {savefile_size_mb:2f} MB')

    